# Notebook to generate files and SPA-C training dataset

In [1]:
import os, time, datetime

import sys
sys.path.append('../py_scripts')

# CHM13

## Splitting chromosomes in subsets

In [2]:
chm13_fa = "/home/amergez/SPA-C/datasets/raw_data/CHM13v2/CHM13v2.fa"
chm13_chrom_sizes = "/home/amergez/SPA-C/datasets/raw_data/CHM13v2/CHM13v2.fa.chrom.sizes"

In [3]:
# Getting chromosome sizes
chrom_sizes = {}
with open(chm13_chrom_sizes, "r") as handle:
    for line in handle.readlines():
        seq_name, length = line.split("\t")
        chrom_sizes[seq_name] = int(length)
chrom_sizes

# Getting the number of "intra" samples per sequence (also filtering small sequences)
samples_per_seq = {seq: (length//5000)-20 for seq, length in chrom_sizes.items() if (length//5000)-20 > 0}
samples_per_seq


{'chr1': 49657,
 'chr2': 48519,
 'chr3': 40201,
 'chr4': 38694,
 'chr5': 36389,
 'chr6': 34405,
 'chr7': 32093,
 'chr8': 29231,
 'chr9': 30103,
 'chr10': 26931,
 'chr11': 27005,
 'chr12': 26644,
 'chr13': 22693,
 'chr14': 20212,
 'chr15': 19930,
 'chr16': 19246,
 'chr17': 16835,
 'chr18': 16088,
 'chr19': 12321,
 'chr20': 13222,
 'chr21': 8998,
 'chr22': 10244,
 'chrX': 30831}

In [4]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

seed=42
val_size = 0.1
test_size = 0.1

x, y, group = [], [], []
cur_grp = 0
for seq_name, n_samples in samples_per_seq.items():
    x.extend([seq_name]*n_samples)
    y.extend([1]*n_samples)
    group.extend([cur_grp]*n_samples)

    cur_grp += 1

x = np.array(x)
y = np.array(y)
group = np.array(group)

In [5]:
# Checking that each chromosome has a single group
for seq_name in np.unique(x):
    mask = (x == seq_name)
    print(seq_name, np.unique(group[mask]))
    

chr1 [0]
chr10 [9]
chr11 [10]
chr12 [11]
chr13 [12]
chr14 [13]
chr15 [14]
chr16 [15]
chr17 [16]
chr18 [17]
chr19 [18]
chr2 [1]
chr20 [19]
chr21 [20]
chr22 [21]
chr3 [2]
chr4 [3]
chr5 [4]
chr6 [5]
chr7 [6]
chr8 [7]
chr9 [8]
chrX [22]


In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=val_size+test_size, random_state=seed)
train_idx, test_val_idx = next(splitter.split(
    x,
    y,
    groups=group)
)

adjusted_val_size = val_size / (val_size+test_size)

splitter_val = GroupShuffleSplit(n_splits=1, test_size=adjusted_val_size, random_state=seed)
test_idx, val_idx = next(splitter_val.split(
    x[test_val_idx],
    y[test_val_idx],
    groups=group[test_val_idx])
)

print(train_idx.shape, test_idx.shape, val_idx.shape)
print(f"Train: {np.unique(x[train_idx])}\nTest: {np.unique(x[test_val_idx][test_idx])}\nVal:{np.unique(x[test_val_idx][val_idx])}")

with open("/home/amergez/SPA-C/datasets/raw_data/CHM13v2/SPLITS.txt", "w") as handle:
    for subset, sequences in {"Train":np.unique(x[train_idx]), "Test":np.unique(x[test_val_idx][test_idx]), "Val":np.unique(x[test_val_idx][val_idx])}.items():
        handle.write(f"{subset}\t{','.join(sequences.tolist())}\n")

(468467,) (68903,) (73122,)
Train: ['chr11' 'chr12' 'chr13' 'chr14' 'chr15' 'chr17' 'chr19' 'chr2' 'chr20'
 'chr21' 'chr22' 'chr3' 'chr4' 'chr5' 'chr6' 'chr7' 'chr8' 'chrX']
Test: ['chr1' 'chr16']
Val:['chr10' 'chr18' 'chr9']


## Generating chimeras

```shell
apps="apptainer" # Path to apptainer images folder
SPA_dir="SPA-C" # Directory of the Git repo
BASE_DIR="datasets"
DS="CHM13v2"
dir="${BASE_DIR}/${DS}" # Directory of CHM13 dataset

# Inter only
mkdir -p ${BASE_DIR}/${DS}.C4A/references
$apps/HiC.sif python -u ${SPA_dir}/model_training/ChimericContigs.py \
  --fasta ${dir}/references/${DS}.fa \
  --splits ${dir}/references/SPLITS.txt \
  --out ${BASE_DIR}/${DS}.C4A/references/${DS}.C4A.fa \
  --seed 42 \
  --paf ${BASE_DIR}/${DS}.C4A/references/${DS}.C4A.paf \
  --repetition 2000 \
  --size 200000 \
  --simple-names "C4A"

# Intra only
mkdir -p ${BASE_DIR}/${DS}.C4B/references
$apps/HiC.sif python -u ${SPA_dir}/model_training/ChimericContigs.py \
  --fasta ${dir}/references/${DS}.fa \
  --splits ${dir}/references/SPLITS.txt \
  --out ${BASE_DIR}/${DS}.C4B/references/${DS}.C4B.fa \
  --seed 42 \
  --paf ${BASE_DIR}/${DS}.C4B/references/${DS}.C4B.paf \
  --repetition 2500 \
  --size 200000 \
  --intra \
  --simple-names "C4B"

# Inversions only
mkdir -p ${BASE_DIR}/${DS}.C4C/references
$apps/HiC.sif python -u ${SPA_dir}/model_training/ChimericContigs.py \
  --fasta ${dir}/references/${DS}.fa \
  --splits ${dir}/references/SPLITS.txt \
  --out ${BASE_DIR}/${DS}.C4C/references/${DS}.C4C.fa \
  --seed 42 \
  --paf ${BASE_DIR}/${DS}.C4C/references/${DS}.C4C.paf \
  --repetition 2500 \
  --size 200000 \
  --inversion \
  --simple-names "C4C"
```